# Module 2: Lists, Recursion & Induction in ACL2

**Prerequisites:** Module 1 (Introduction to ACL2)

In this module we explore ACL2's fundamental data structure — the **cons pair** — and learn how recursive functions over lists lead naturally to **inductive proofs**. Every recursive `defun` in ACL2 automatically suggests an induction scheme that the theorem prover can exploit.

**Learning Objectives:**
1. Understand cons pairs, `car`, `cdr`, and list construction
2. Write recursive functions with termination measures
3. Prove theorems by induction on list structure
4. Master the accumulator pattern and prove its correctness
5. Work with trees as nested cons structures

## 1. Cons Pairs: The Building Block

Every non-atomic ACL2 object is a **cons pair** — a pair `(a . b)` built with `cons`, decomposed with `car` (first) and `cdr` (rest). Lists are just cons pairs whose final `cdr` is `nil`.

| Expression | Value | Note |
|---|---|---|
| `(cons 1 2)` | `(1 . 2)` | A dotted pair |
| `(cons 1 nil)` | `(1)` | A one-element list |
| `(cons 1 (cons 2 nil))` | `(1 2)` | A two-element list |
| `(car '(a b c))` | `a` | First element |
| `(cdr '(a b c))` | `(b c)` | Rest of list |

In [ ]:
; Build cons pairs and explore car/cdr
(cons 1 2)

In [ ]:
; A list is a chain of cons pairs ending in nil
(cons 'a (cons 'b (cons 'c nil)))

In [ ]:
; Shorthand — quote notation
'(a b c)

In [ ]:
; Decomposing
(car '(a b c))

In [ ]:
(cdr '(a b c))

In [ ]:
; Predicates
(consp '(1 2))   ; t - is it a cons pair?


In [ ]:
(atom 42)        ; t - is it an atom?


In [ ]:
(endp nil)       ; t - is the list empty?

## 2. Recursive Functions on Lists

Recursive functions on lists follow a standard pattern:
1. **Base case:** Handle `endp` (empty list) or `atom`
2. **Recursive case:** Process `(car x)`, recurse on `(cdr x)`

ACL2 requires every recursive function to **terminate**. By default, ACL2 uses `acl2-count` as the measure and verifies it decreases on each recursive call.

In [ ]:
; Length of a list
(defun my-len (x)
  (if (endp x)
      0
    (+ 1 (my-len (cdr x)))))

In [ ]:
(my-len '(a b c d))

In [ ]:
; Append two lists
(defun my-app (x y)
  (if (endp x)
      y
    (cons (car x)
          (my-app (cdr x) y))))

In [ ]:
(my-app '(1 2) '(3 4 5))

In [ ]:
; Reverse a list (naive quadratic approach)
(defun my-rev (x)
  (if (endp x)
      nil
    (my-app (my-rev (cdr x))
            (list (car x)))))

In [ ]:
(my-rev '(a b c d))

In [ ]:
; Membership test
(defun my-mem (e x)
  (if (endp x)
      nil
    (if (equal e (car x))
        t
      (my-mem e (cdr x)))))

In [ ]:
(my-mem 'b '(a b c))

## 3. Proof by Induction

When you define a recursive function `f` that recurs on `(cdr x)`, ACL2 automatically creates an **induction scheme**:
- **Base case:** `(atom x)` — the list is empty or not a cons
- **Induction step:** Assume the theorem holds for `(cdr x)`, prove it for `x`

This is **structural induction** on lists, directly mirroring the recursive structure of the function.

### Key insight
> *Every recursive definition suggests an induction. The case analysis mirrors the function's termination argument.*

Let's prove some fundamental properties.

In [ ]:
; Theorem: appending nil does nothing
(defthm my-app-nil
  (equal (my-app x nil)
         (true-list-fix x)))

The theorem above uses `true-list-fix` because `my-app` preserves the final `cdr` of `x`. For true lists, `(true-list-fix x) = x`.

In [ ]:
; Theorem: append is associative  
; This is foundational — from books/textbook/chap10/tree.lisp
(defthm my-app-assoc
  (equal (my-app (my-app x y) z)
         (my-app x (my-app y z))))

ACL2 proved `my-app-assoc` automatically by induction on `x`. The proof follows the recursion of `my-app`:
- **Base:** When `(endp x)`, both sides reduce to `(my-app y z)` ✓
- **Step:** Assume the theorem for `(cdr x)`. Then `(my-app (my-app x y) z)` = `(cons (car x) (my-app (my-app (cdr x) y) z))` = (by IH) = `(cons (car x) (my-app (cdr x) (my-app y z)))` = `(my-app x (my-app y z))` ✓

In [ ]:
; Theorem: length distributes over append
(defthm my-len-of-my-app
  (equal (my-len (my-app x y))
         (+ (my-len x) (my-len y))))

## 4. When Induction Needs Help: Lemmas

Some theorems require **lemmas** — helper theorems proved first. This is analogous to breaking a mathematical proof into smaller steps.

Let's prove that reversing a list preserves its length. We need a lemma about `my-app` first.

In [ ]:
; Lemma: length of reverse
(defthm my-len-of-my-rev
  (equal (my-len (my-rev x))
         (my-len x)))

ACL2 proves this using the previously established `my-len-of-my-app` as a rewrite rule. The **lemma stacking** pattern is central to ACL2 proof development:

```
my-app-assoc  ────┐
my-app-nil    ────┤
my-len-of-my-app ─┼──> my-len-of-my-rev
```

## 5. The Accumulator Pattern

The naive `my-rev` is $O(n^2)$ because it calls `my-app` at each step. A faster approach uses an **accumulator** — an extra parameter that builds the result as we go.

This pattern appears in the ACL2 textbook (Chapter 10) with the classic `flatten` / `mc-flatten` example.

### From `books/textbook/chap10/tree.lisp`:

In [ ]:
; Tail-recursive reverse with accumulator
(defun my-rev-acc (x acc)
  (if (endp x)
      acc
    (my-rev-acc (cdr x)
                (cons (car x) acc))))

In [ ]:
(my-rev-acc '(1 2 3 4) nil)

Now we must prove that `my-rev-acc` computes the same result as `my-rev`. The standard technique:
1. Prove a **generalized lemma** relating `my-rev-acc` to `my-rev` and `my-app`
2. Derive the desired theorem as a corollary

In [ ]:
; The key generalization lemma
(defthm my-rev-acc-is-app
  (equal (my-rev-acc x acc)
         (my-app (my-rev x) acc)))

In [ ]:
; Corollary: my-rev-acc with nil accumulator = my-rev
(defthm my-rev-acc-correct
  (equal (my-rev-acc x nil)
         (my-rev x)))

## 6. Trees as Nested Cons Structures

In ACL2, **trees** are simply nested cons pairs where leaves are atoms. The textbook's `flatten` / `mc-flatten` example (from `books/textbook/chap10/tree.lisp`) illustrates tree recursion and the accumulator pattern beautifully.

In [ ]:
; Flatten a tree into a list of its leaves
; From books/textbook/chap10/tree.lisp
(defun my-flatten (x)
  (cond ((atom x) (list x))
        (t (append (my-flatten (car x))
                   (my-flatten (cdr x))))))

In [ ]:
(my-flatten '((1 . 2) . (3 . (4 . 5))))

In [ ]:
; Accumulator-based version (tail-recursive)
; From books/textbook/chap10/tree.lisp
(defun my-mc-flatten (x a)
  (cond ((atom x) (cons x a))
        (t (my-mc-flatten (car x)
                          (my-mc-flatten (cdr x) a)))))

In [ ]:
(my-mc-flatten '((1 . 2) . (3 . (4 . 5))) nil)

### Proving the Accumulator Version Correct

We need to show `(my-mc-flatten x nil) = (my-flatten x)`. As before, we prove a generalization first.

In [ ]:
; Generalized lemma: mc-flatten relates to flatten + append
(defthm my-mc-flatten-is-append
  (equal (my-mc-flatten x a)
         (append (my-flatten x) a)))

In [ ]:
; Correctness theorem
(defthm my-mc-flatten-correct
  (equal (my-mc-flatten x nil)
         (my-flatten x)))

## 7. Classic List Theorems

The ACL2 standard library (`books/std/lists/`) contains hundreds of theorems about lists. Here are some fundamental ones you should know, which we can prove ourselves.

In [ ]:
; Reverse of reverse is the original (for true-lists)
(defun my-true-listp (x)
  (if (consp x)
      (my-true-listp (cdr x))
    (equal x nil)))

In [ ]:
; Reverse distributes over append (in reverse order)
(defthm my-rev-of-my-app
  (equal (my-rev (my-app x y))
         (my-app (my-rev y) (my-rev x))))

In [ ]:
; A rewrite rule needed for rev-rev
(defthm my-app-nil-v2
  (implies (my-true-listp x)
           (equal (my-app x nil) x)))

In [ ]:
; Reverse of reverse
(defthm my-rev-of-my-rev
  (implies (my-true-listp x)
           (equal (my-rev (my-rev x)) x)))

## 8. Termination and Measures

ACL2 requires all functions to terminate on all inputs. By default, it uses `acl2-count` — a built-in measure that assigns a natural number to any ACL2 object.

| Object | `acl2-count` |
|---|---|
| Integer $n$ | $|n|$ |
| Rational $p/q$ | $|p| + |q|$ |
| Cons pair `(a . b)` | $1 + $ `acl2-count(a)` $+$ `acl2-count(b)` |
| Symbol, String, Char | $0$ |

For functions that don't recur on `cdr`, you may need a custom `:measure`.

In [ ]:
; Example: Collatz-like function with explicit measure
(defun ack (m n)
  (declare (xargs :measure (+ (acl2-count m) (acl2-count n))))
  (cond ((zp m) (+ n 1))
        ((zp n) (ack (- m 1) 1))
        (t (ack (- m 1) (ack m (- n 1))))))

Wait — the Ackermann function actually needs a **lexicographic** measure. Let's use the approach ACL2 supports:

In [ ]:
; ACL2 uses ordinals for termination.
; For Ackermann, we need a two-component measure:
(defun ack2 (m n)
  (declare (xargs :measure (make-ord 1 (1+ (nfix m)) (1+ (nfix n)))))
  (cond ((zp m) (+ n 1))
        ((zp n) (ack2 (- m 1) 1))
        (t (ack2 (- m 1) (ack2 m (- n 1))))))

In [ ]:
(ack2 3 4)  ; = 125

## 9. The Standard List Library

The ACL2 community books at `books/std/lists/` provide verified implementations and hundreds of theorems for common list operations:

| Book | Key Contents |
|---|---|
| `append.lisp` | `len-of-append`, `append-of-nil`, associativity |
| `rev.lisp` | `rev` function, `rev-of-append`, involutive property |
| `len.lisp` | `len-when-atom`, `len-of-cons`, positivity |
| `nth.lisp` | Indexed access theorems |
| `take.lisp` | First-n-elements with properties |
| `nthcdr.lisp` | Drop-first-n with `append-take-nthcdr` |
| `member.lisp` | Membership predicates |
| `flatten.lisp` | Guard-free concatenation and flattening |
| `duplicity.lisp` | Element counting (occurrences) |

Use `(include-book "std/lists/top" :dir :system)` to load all of them at once.

## 10. Exercises

**Exercise 1:** Define `my-last` that returns the last element of a non-empty list. Prove that `(my-last (my-app x (list e))) = e`.

**Exercise 2:** Define `my-remove` that removes all occurrences of element `e` from list `x`. Prove that `(my-mem e (my-remove e x)) = nil`.

**Exercise 3 (Accumulator):** Define `my-flatten-acc` — a tail-recursive version of `my-flatten` using an accumulator. Prove it correct with respect to `my-flatten`.

**Exercise 4 (Challenge):** Define a function `my-zip` that interleaves two lists: `(my-zip '(a b) '(1 2))` = `(a 1 b 2)`. Prove `(my-len (my-zip x y)) = (+ (my-len x) (my-len y))` when the lists have equal length.

---
**Next:** [Module 3 — Sorting Algorithms](03_sorting.ipynb) — We verify insertion sort, merge sort, and their equivalence.